# Portable Storage Setup Check

Use this notebook first on Middle Earth or Rubin Science Platform. It reports the complete effective storage configuration before any write, performs a policy-aware preflight, and optionally runs a tiny LSST-only ANTARES probe before a long backfill. Set storage environment variables before importing `src.config`; restart the kernel after changing them. Cache creation is disabled unless you explicitly opt in below.

In [ ]:
from pathlib import Path
import os
import sys

# Make imports robust whether Jupyter starts in the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
candidate = Path.home() / "notebooks" / "ANTARES_Analysis"
if not (PROJECT_ROOT / "src").exists() and (candidate / "src").exists():
    PROJECT_ROOT = candidate
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, history, query, rsp_permissions

DATA_ROOT = config.DATA_ROOT
CACHE_ROOT = config.CACHE_ROOT
LSST_DATA_ROOT = config.LSST_ONLY_ROOT
NIGHTLY_ROOT = config.NIGHTLY_ROOT
CUMULATIVE_ROOT = config.CUMULATIVE_ROOT
ANALYSIS_ROOT = config.ANALYSIS_ROOT
STORAGE_POLICY = config.STORAGE_POLICY
SHARED_GROUP = config.EXPECTED_SHARED_GROUP if STORAGE_POLICY == "shared-group" else None
SCRATCH_DIR = os.getenv("SCRATCH_DIR")
SCRATCH_ROOT = Path(SCRATCH_DIR).expanduser() / "ANTARES_Analysis" if SCRATCH_DIR else None

# Merely configuring CACHE_ROOT never authorizes creating or warming it.
CREATE_CACHE_ROOT = False

print(f"Project root : {PROJECT_ROOT}")
print(f"HOME         : {os.getenv('HOME')}")
print(f"USER         : {os.getenv('USER') or os.getenv('JUPYTERHUB_USER')}")
print(f"Scratch root : {SCRATCH_ROOT if SCRATCH_ROOT is not None else 'not configured'}")
print(f"Data root    : {DATA_ROOT}")
print(f"Cache root   : {CACHE_ROOT}")
print(f"Nightly root : {NIGHTLY_ROOT}")
print(f"Cumulative   : {CUMULATIVE_ROOT}")
print(f"Analysis root: {ANALYSIS_ROOT}")
print(f"Storage policy: {STORAGE_POLICY}")
print(f"Shared group : {SHARED_GROUP if SHARED_GROUP is not None else 'not used'}")
print(f"Create cache : {CREATE_CACHE_ROOT}")
print("Restart this kernel after changing storage environment variables.")

rsp_permissions.configure_process_umask(policy=STORAGE_POLICY)


In [ ]:
import pandas as pd
import pyarrow
from antares_client.search import search as antares_search

print("Imports complete.")
print(f"pandas  : {pd.__version__}")
print(f"pyarrow : {pyarrow.__version__}")
config.print_config_summary()


In [ ]:
# Read-only policy preflight. Directory creation remains in the explicit cell below.
preflight_report = rsp_permissions.require_storage_root(
    DATA_ROOT,
    cache_root=CACHE_ROOT,
    policy=STORAGE_POLICY,
    expected_group=SHARED_GROUP,
    write_test=False,
)


In [ ]:
storage_paths = [
    NIGHTLY_ROOT,
    CUMULATIVE_ROOT,
    config.LSST_ONLY_ROOT / "analysis",
    ANALYSIS_ROOT,
    DATA_ROOT / "logs",
]
for path in storage_paths:
    rsp_permissions.ensure_storage_path(
        path, policy=STORAGE_POLICY, expected_group=SHARED_GROUP
    )
    print(f"OK: {path}")

if CREATE_CACHE_ROOT:
    rsp_permissions.ensure_storage_path(
        CACHE_ROOT, policy=STORAGE_POLICY, expected_group=SHARED_GROUP
    )
    print(f"Cache root created/verified by explicit opt-in: {CACHE_ROOT}")
else:
    print(f"Cache creation disabled; configured path was not created: {CACHE_ROOT}")

if SCRATCH_ROOT is not None:
    SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Temporary scratch: {SCRATCH_ROOT}")
else:
    print("Temporary scratch: not configured; no scratch directory was created")

print(f"Persistent LSST store: {LSST_DATA_ROOT}")
print(f"Configured cache root: {CACHE_ROOT}")


In [ ]:
probe = query.query_range(
    label='LSST-only setup probe',
    mjd_min=config.LSST_HISTORY_START_MJD,
    mjd_max=config.LSST_HISTORY_START_MJD + 1,
    n_samples=5,
    tag=config.QUERY_TAG,
    seed=None,
    verbose=True,
    lsst_only=True,
)

counts = query.lsst_identifier_counts(probe)
print(counts)
if not probe.empty:
    assert counts['lsst_identifier_count'] == len(probe), 'Probe returned non-LSST loci.'
    display_cols = [col for col in ['locus_id', 'ra', 'dec', 'newest_alert_observation_time', 'survey', 'ztf_object_id'] if col in probe.columns]
    display(probe[display_cols].head())
else:
    print('Probe returned 0 rows. Try a later MJD window before running backfill.')